In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
random.seed(666)

anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles_balanced.csv'

In [12]:
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import ContraSeqDataset, get_dataset_array
from losses import SupConLoss, padce_loss
from tqdm.notebook import tqdm

import torch
# from tqdm.auto import trange, tqdm

torch.cuda.empty_cache()
use_cuda = True
device =  torch.device("cuda" if use_cuda else "cpu")

## Dataset
ds = ContraSeqDataset(anc_path, aug_path)
ds_arr = get_dataset_array(anc_path, aug_path)
anc_map = get_anc_map(ds_arr)

In [13]:
## Model instantiation
model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)

supcon_loss = SupConLoss()

lr = 0.00001
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = None 

model = model.to(device)
model = model.train()

In [26]:
from contra_seq_dataset import AnchoredSampler
from torch.utils.data import DataLoader, RandomSampler
from contra_seq_dataset import ContraSeqDataset, get_anc_map

bs = 3 # this is the number of ANCHORS

sampler = AnchoredSampler(sampler = RandomSampler(list(anc_map.keys())), 
                          anc_map = anc_map, 
                          batch_size = bs, 
                          drop_last = True)

loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, pin_memory=True)

for samp in loader:
    print(len(samp['smiles']))
    print(sum([1 for x in samp['atype'] if x=='Anc']))
    print(sum([1 for x in samp['atype'] if x=='Aug']))
    break

18
3
15


In [ ]:
from contra_seq_dataset import AnchoredSampler
from torch.utils.data import DataLoader, RandomSampler
from contra_seq_dataset import ContraSeqDataset, get_anc_map

bs = 3 # this is the number of ANCHORS

sampler = AnchoredSampler(sampler = RandomSampler(list(anc_map.keys())), 
                          anc_map = anc_map, 
                          batch_size = bs, 
                          drop_last = True)

loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, pin_memory=True)

for samp in loader:
    print(len(samp['smiles']))
    print(sum([1 for x in samp['atype'] if x=='Anc']))
    print(sum([1 for x in samp['atype'] if x=='Aug']))
    break

In [ ]:
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import ContraSeqDataset

loader = DataLoader(ds, bs, shuffle=True, num_workers=0, pin_memory=True)

for samp in tqdm(loader,total=(len(ds)//bs)):
    
#     print(samp)
#     break
    
    optimizer.zero_grad()
    
    latent = []
    for fuzz_ball in samp:
#         print(fuzz_ball)
        for k,v in fuzz_ball.items():
            if torch.is_tensor(v):
                fuzz_ball[k] = v.to(device)
        latent_vec, dec_out = model.forward(fuzz_ball['seq'],
                                            fuzz_ball['pad_mask'], 
                                            fuzz_ball['avg_mask'], 
                                            fuzz_ball['out_mask'])
        latent.append(latent_vec)
    latent = torch.cat(latent)
    latent = latent.unsqueeze(1)

    contra_loss = supcon_loss(latent, labels=labels)
    dec_out = dec_out.squeeze()
    print(dec_out.shape)
#     recon_loss = padce_loss(fuzz_ball['seq'],dec_out,fuzz_ball['pad_mask'],
#                             fuzz_ball['out_mask'])
    
    print(contra_loss.item()) #, recon_loss.item())
    loss = contra_loss # + recon_loss
    loss.backward()
    optimizer.step()
torch.cuda.empty_cache()

